# Qwen3 Layer Export - Export Model with Layer Hidden States

This notebook exports the Qwen3-4B model with exposed layer outputs for EVA-Ai FCP system.

**Output files:**
- `qwen_layer_model.pt` - PyTorch model with layer hooks

In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

# Output directory
OUTPUT_DIR = '/content/drive/MyDrive/EVA_Ai_Exports'
!mkdir -p $OUTPUT_DIR

In [ ]:
# Install dependencies
!pip install -q transformers torch

In [ ]:
import os
import torch
from transformers import AutoModelForCausalLM, AutoConfig, AutoTokenizer
import logging

logging.basicConfig(level=logging.INFO)
logger = logging.getLogger('export')

In [ ]:
# Model configuration
MODEL_PATH = 'RefalMachine/RuadaptQwen3-4B-Instruct'
NUM_LAYERS = 36
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

logger.info(f"Using device: {DEVICE}")

In [ ]:
# Load model
logger.info(f"Loading model from {MODEL_PATH}")

config = AutoConfig.from_pretrained(
    MODEL_PATH,
    trust_remote_code=True
)
logger.info(f"  hidden_size: {config.hidden_size}")
logger.info(f"  num_layers: {config.num_hidden_layers}")

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_PATH,
    trust_remote_code=True
)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL_PATH,
    config=config,
    trust_remote_code=True,
    torch_dtype=torch.bfloat16,
    device_map=DEVICE,
    low_cpu_mem_usage=True
)

model.eval()
logger.info(f"Model loaded successfully")

In [ ]:
# Model wrapper with layer hooks

class ModelWithLayerOutputs(torch.nn.Module):
    def __init__(self, base_model, num_layers):
        super().__init__()
        self.base_model = base_model
        self.num_layers = num_layers
        self.layer_outputs = {}
        
        for idx, layer in enumerate(base_model.model.layers):
            layer.register_forward_hook(self._hook_fn(idx))

    def _hook_fn(self, layer_idx):
        def hook(module, input, output):
            self.layer_outputs[layer_idx] = output[0].detach()
        return hook

    def forward(self, input_ids, attention_mask=None):
        self.layer_outputs.clear()
        
        outputs = self.base_model(
            input_ids=input_ids,
            attention_mask=attention_mask
        )

        all_outputs = {}
        all_outputs[-1] = self.base_model.model.embed_tokens(input_ids).detach()
        
        for idx in range(self.num_layers):
            if idx in self.layer_outputs:
                all_outputs[idx] = self.layer_outputs[idx]
        
        all_outputs['final'] = outputs.logits
        
        return all_outputs

In [ ]:
# Create wrapped model
wrapped_model = ModelWithLayerOutputs(model, NUM_LAYERS)
wrapped_model.eval()

# Test inference
test_input = tokenizer("Привет", return_tensors="pt").to(DEVICE)
with torch.no_grad():
    outputs = wrapped_model(test_input['input_ids'], test_input['attention_mask'])

print(f"Embedding output shape: {outputs[-1].shape}")
for layer_idx in sorted([k for k in outputs.keys() if isinstance(k, int)])[:5]:
    print(f"Layer {layer_idx} output shape: {outputs[layer_idx].shape}")
print(f"...")
print(f"Final logits shape: {outputs['final'].shape}")

In [ ]:
# Save PyTorch model
pt_path = os.path.join(OUTPUT_DIR, 'qwen_layer_model.pt')

torch.save({
    'model_state_dict': wrapped_model.state_dict(),
    'config': config,
    'num_layers': NUM_LAYERS
}, pt_path)

logger.info(f"PyTorch model saved: {pt_path}")
!ls -lh $pt_path

In [ ]:
# List exported files
print("\nExported files:")
!ls -lh $OUTPUT_DIR